### Add candidate molecule names

In [2]:
# enrich_candidates_via_ds_results.py
# ------------------------------------------------------------
# Robust candidate-name enrichment for a METASPACE manifest:
#   - Uses ds.results(...) (supported by the Python client)
#   - Pulls moleculeNames for (sumFormula, adduct) pairs
#   - Avoids fragile direct GraphQL datasetName filtering
# ------------------------------------------------------------

from __future__ import annotations
from pathlib import Path
import re, json, time
import pandas as pd
from metaspace import SMInstance
from tqdm import tqdm

# ------------------ CONFIG ------------------
MANIFEST_IN  = Path(r"metaspace_images_dump\manifest_REBUILT2_expanded_spectral_only.parquet")
MANIFEST_OUT = Path(r"metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates.parquet")
CACHE_JSON   = Path(r"metaspace_images_dump\cand_names_cache_ds_results.json")

METASPACE_HOST = "https://metaspace2020.eu"

# If you want to restrict to the dataset's DBs, leave DB=None (default).
# If you want to force a specific db, set one of:
#   DB = ("HMDB", "v4")   # tuple style
#   DB = "HMDB-v4"        # string style
DB = None

# fdr can be None (all results), or a float to filter
FDR_MAX = None

# Parse ".../images/{FORMULA}_{ADDUCT}/peak0.npz"
ANN_DIR_RE = re.compile(
    r"[\\/](images)[\\/](?P<formula>[A-Za-z0-9]+)_(?P<adduct>[^\\/]+)[\\/]",
    re.IGNORECASE
)

def _bad(x) -> bool:
    if x is None:
        return True
    s = str(x).strip()
    if s == "" or "NAME" in s.upper():  # Excel "#NAME?"
        return True
    return False

def derive_formula_adduct(row) -> tuple[str | None, str | None]:
    sf = row.get("sum_formula")
    ad = row.get("adduct")

    if _bad(sf) or _bad(ad):
        m = ANN_DIR_RE.search(str(row.get("path", "")))
        if m:
            sf = m.group("formula")
            ad = m.group("adduct")

    sf = str(sf).strip() if sf is not None else None
    ad = str(ad).strip() if ad is not None else None
    return sf, ad

def _normalize_adduct(a: str) -> str:
    # ds.results index adducts look like "+K", "+H", "-H" etc.
    # just be safe about whitespace
    return (a or "").strip().replace(" ", "")

def _dedup_preserve(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def load_results_for_dataset(ds, database=None, fdr_max=None) -> pd.DataFrame:
    """
    Returns ds.results(...) as a DataFrame with (formula, adduct) index and moleculeNames column.
    """
    if database is None and fdr_max is None:
        res = ds.results()
    elif database is None:
        res = ds.results(fdr=fdr_max)
    elif fdr_max is None:
        res = ds.results(database=database)
    else:
        res = ds.results(database=database, fdr=fdr_max)

    # Ensure DataFrame
    if not isinstance(res, pd.DataFrame):
        res = pd.DataFrame(res)

    return res

def main():
    df = pd.read_parquet(MANIFEST_IN)

    for col in ["path", "dataset_id", "name"]:
        if col not in df.columns:
            raise ValueError(f"Manifest must include '{col}' column.")

    # Clean formula/adduct
    sfs, ads = [], []
    for _, r in df.iterrows():
        sf, ad = derive_formula_adduct(r)
        sfs.append(sf)
        ads.append(ad)
    df["sum_formula_clean"] = sfs
    df["adduct_clean"] = ads

    rep = (
        df[["sum_formula_clean", "adduct_clean", "dataset_id"]]
        .dropna()
        .drop_duplicates(subset=["sum_formula_clean", "adduct_clean", "dataset_id"])
        .reset_index(drop=True)
    )

    print(f"[INFO] unique (dataset, formula, adduct): {len(rep)}")

    cache = json.loads(CACHE_JSON.read_text()) if CACHE_JSON.exists() else {}

    sm = SMInstance(host=METASPACE_HOST)
    # If you need private datasets:
    # sm.save_login()

    # We'll process per dataset_id to avoid re-downloading results repeatedly
    by_ds = rep.groupby("dataset_id", sort=False)

    for dsid, sub in tqdm(by_ds, desc="Datasets", unit="ds"):
        dsid = str(dsid)

        # Load dataset
        try:
            ds = sm.dataset(id=dsid)
        except Exception as e:
            # Cache failures as empty for all pairs in this dataset
            for sf, ad in sub[["sum_formula_clean", "adduct_clean"]].itertuples(index=False):
                k = f"{dsid}||{sf}||{_normalize_adduct(ad)}"
                cache[k] = {"dataset_id": dsid, "sum_formula": sf, "adduct": _normalize_adduct(ad), "names": []}
            continue

        # Pull results once
        try:
            res = load_results_for_dataset(ds, database=DB, fdr_max=FDR_MAX)
        except Exception:
            for sf, ad in sub[["sum_formula_clean", "adduct_clean"]].itertuples(index=False):
                k = f"{dsid}||{sf}||{_normalize_adduct(ad)}"
                cache[k] = {"dataset_id": dsid, "sum_formula": sf, "adduct": _normalize_adduct(ad), "names": []}
            continue

        # We need moleculeNames; some client versions use 'moleculeNames' exactly (as in examples),
        # but be defensive.
        name_col = None
        for c in ["moleculeNames", "molecule_names", "moleculeName", "possibleCompounds"]:
            if c in res.columns:
                name_col = c
                break
        if name_col is None:
            # nothing to extract
            for sf, ad in sub[["sum_formula_clean", "adduct_clean"]].itertuples(index=False):
                k = f"{dsid}||{sf}||{_normalize_adduct(ad)}"
                cache[k] = {"dataset_id": dsid, "sum_formula": sf, "adduct": _normalize_adduct(ad), "names": []}
            continue

        # Build lookup from index -> names
        # Index is usually MultiIndex (formula, adduct)
        lookup = {}
        if isinstance(res.index, pd.MultiIndex) and res.index.nlevels >= 2:
            for (sf_i, ad_i), row in res.iterrows():
                sf_i = str(sf_i).strip()
                ad_i = _normalize_adduct(str(ad_i))
                val = row.get(name_col)
                if isinstance(val, (list, tuple)):
                    names = [str(x) for x in val if x]
                elif pd.isna(val):
                    names = []
                else:
                    # sometimes it's a string like "[A, B, C]" or "A; B"
                    s = str(val)
                    if s.startswith("[") and s.endswith("]"):
                        s = s[1:-1]
                    names = [t.strip().strip("'\"") for t in re.split(r"[;,]\s*|\s*\|\s*", s) if t.strip()]
                lookup[(sf_i, ad_i)] = _dedup_preserve(names)
        else:
            # Fallback if index shape differs: try formula/adduct columns
            if "formula" in res.columns and "adduct" in res.columns:
                for _, row in res.iterrows():
                    sf_i = str(row["formula"]).strip()
                    ad_i = _normalize_adduct(str(row["adduct"]))
                    val = row.get(name_col)
                    names = val if isinstance(val, list) else []
                    lookup[(sf_i, ad_i)] = _dedup_preserve([str(x) for x in names if x])

        # Fill cache for requested pairs in this dataset
        for sf, ad in sub[["sum_formula_clean", "adduct_clean"]].itertuples(index=False):
            sf = str(sf).strip()
            ad = _normalize_adduct(str(ad))
            k = f"{dsid}||{sf}||{ad}"
            if k in cache:
                continue
            names = lookup.get((sf, ad), [])
            cache[k] = {"dataset_id": dsid, "sum_formula": sf, "adduct": ad, "names": names}

        # periodic flush
        if len(cache) % 100 == 0:
            CACHE_JSON.write_text(json.dumps(cache, indent=2))

        # be polite to the server
        time.sleep(0.05)

    CACHE_JSON.write_text(json.dumps(cache, indent=2))
    print(f"[CACHE] wrote {CACHE_JSON} ({len(cache)} entries)")

    join_df = pd.DataFrame([
        {
            "dataset_id": rec["dataset_id"],
            "sum_formula_clean": rec["sum_formula"],
            "adduct_clean": rec["adduct"],
            "cand_names": "; ".join(rec["names"]),
        }
        for rec in cache.values()
    ])

    out = df.merge(join_df, on=["dataset_id", "sum_formula_clean", "adduct_clean"], how="left")
    out.to_parquet(MANIFEST_OUT, index=False)
    print(f"[OK] wrote {MANIFEST_OUT}")

if __name__ == "__main__":
    main()


[INFO] unique (dataset, formula, adduct): 231884


Datasets: 100%|██████████| 7653/7653 [2:59:57<00:00,  1.41s/ds]   


[CACHE] wrote metaspace_images_dump\cand_names_cache_ds_results.json (231884 entries)
[OK] wrote metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates.parquet


In [1]:
import pandas as pd
pd.read_parquet("metaspace_images_dump/manifest_REBUILT2_expanded_with_candidates.parquet")

,dataset_id,name,organism,split,db,fdr,msm,sum_formula,adduct,isotope_index,...,max_isotopes,save_format,tiling,tile_size,tile_stride,Organism_Part,Condition,sum_formula_clean,adduct_clean,cand_names
0,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,0.0,...,NaN,None,None,NaN,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...
1,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,1.0,...,NaN,None,None,NaN,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...
2,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,2.0,...,NaN,None,None,NaN,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...
3,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957267,C10H20O2,-H,0.0,...,NaN,None,None,NaN,NaN,Colon,"Cancer, xenograft",C10H20O2,-H,"(1R,2S,3S,4R)-p-Menthane-2,3-diol; p-Menthane-..."
4,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957267,C10H20O2,-H,1.0,...,NaN,None,None,NaN,NaN,Colon,"Cancer, xenograft",C10H20O2,-H,"(1R,2S,3S,4R)-p-Menthane-2,3-diol; p-Menthane-..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695495,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.867454,C45H78O2,+Na,1.0,...,NaN,None,None,NaN,NaN,Kidney,Control,C45H78O2,+Na,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...
695496,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.867454,C45H78O2,+Na,2.0,...,NaN,None,None,NaN,NaN,Kidney,Control,C45H78O2,+Na,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...
695497,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.899866,C47H95N2O6P,+Na,0.0,...,NaN,None,None,NaN,NaN,Kidney,Control,C47H95N2O6P,+Na,SM(d18:0/24:1(15Z))
695498,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.899866,C47H95N2O6P,+Na,1.0,...,NaN,None,None,NaN,NaN,Kidney,Control,C47H95N2O6P,+Na,SM(d18:0/24:1(15Z))


### Add SMILES

In [2]:
# enrich_structures_from_names_pubchem_offline_streaming2.py
# ------------------------------------------------------------
# OFFLINE PubChem structure enrichment (STREAMING, constant-memory)
# - Reads your manifest parquet ONLY to collect unique cand_names
# - Scans PubChem local files to resolve: name -> CID -> SMILES/InChIKey
# - Updates / writes the JSON cache
#
# Output:
#   metaspace_images_dump\name_to_pubchem_struct_cache.json

from __future__ import annotations
from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm

# ===================== PATHS =====================
IN_PARQUET  = Path(r"metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates.parquet")
CACHE_JSON  = Path(r"metaspace_images_dump\name_to_pubchem_struct_cache.json")

PUBCHEM_DIR  = Path(r"Y:\coskun-lab\Efe\MSI Foundation Model\CID")
FILE_SYNONYM = PUBCHEM_DIR / "CID-Synonym-filtered"
FILE_SMILES  = PUBCHEM_DIR / "CID-SMILES"
FILE_INCHI   = PUBCHEM_DIR / "CID-InChI-Key"

# ===================== CONFIG =====================
# If True: do NOT redo names that already exist in cache
SKIP_CACHED = True

# ===================== HELPERS =====================
def normalize_name(s: str) -> str:
    return s.strip().lower()

def load_cache() -> dict:
    if CACHE_JSON.exists():
        return json.loads(CACHE_JSON.read_text(encoding="utf-8"))
    return {}

def save_cache(cache: dict) -> None:
    CACHE_JSON.parent.mkdir(parents=True, exist_ok=True)
    CACHE_JSON.write_text(json.dumps(cache, indent=2), encoding="utf-8")

# ===================== MAIN =====================
def main():
    # 0) Load manifest (only to read cand_names)
    df = pd.read_parquet(IN_PARQUET)
    if "cand_names" not in df.columns:
        raise ValueError("Expected column 'cand_names'")

    cache = load_cache()

    # Parse candidate name lists
    names_series = (
        df["cand_names"]
        .fillna("")
        .astype(str)
        .map(lambda s: [x.strip() for x in s.split(";") if x.strip()])
    )

    unique_names = sorted({n for lst in names_series for n in lst})
    print(f"[INFO] unique candidate names: {len(unique_names):,}")

    if SKIP_CACHED:
        todo = [n for n in unique_names if n not in cache]
    else:
        todo = unique_names

    print(f"[INFO] names to resolve (this run): {len(todo):,}")
    if not todo:
        print("[INFO] nothing to do (all names already cached).")
        return

    # ---- Step 1: build small target set ----
    # map normalized -> original (keeps one representative original string)
    target_set = {normalize_name(n): n for n in todo}
    found_cids: dict[str, int] = {}  # normalized_name -> CID

    # ---- Step 2: scan synonyms (name -> CID) ----
    print("[STREAM] scanning CID-Synonym-filtered ...")
    with FILE_SYNONYM.open("r", encoding="utf8", errors="ignore") as f:
        for line in tqdm(f):
            try:
                cid_str, name = line.rstrip("\n").split("\t", 1)
            except ValueError:
                continue
            key = normalize_name(name)
            if key in target_set and key not in found_cids:
                try:
                    found_cids[key] = int(cid_str)
                except ValueError:
                    continue
            if len(found_cids) == len(target_set):
                break

    print(f"[OK] matched names: {len(found_cids):,}/{len(target_set):,}")

    needed_cids = set(found_cids.values())
    print(f"[INFO] unique CIDs needed: {len(needed_cids):,}")

    # ---- Step 3: scan SMILES (CID -> SMILES) ----
    cid_to_smiles: dict[int, str] = {}
    print("[STREAM] scanning CID-SMILES ...")
    with FILE_SMILES.open("r", encoding="utf8", errors="ignore") as f:
        for line in tqdm(f):
            try:
                cid_str, sm = line.rstrip("\n").split("\t", 1)
            except ValueError:
                continue
            try:
                cid = int(cid_str)
            except ValueError:
                continue
            if cid in needed_cids:
                cid_to_smiles[cid] = sm
                if len(cid_to_smiles) == len(needed_cids):
                    break

    print(f"[OK] SMILES found: {len(cid_to_smiles):,}/{len(needed_cids):,}")

    # ---- Step 4: scan InChIKeys (CID -> InChIKey) ----
    # File format can be: CID \t InChIKey \t InChI (or more)
    cid_to_inchi: dict[int, str] = {}
    print("[STREAM] scanning CID-InChI-Key ...")
    with FILE_INCHI.open("r", encoding="utf8", errors="ignore") as f:
        for line in tqdm(f):
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            try:
                cid = int(parts[0])
            except ValueError:
                continue
            if cid in needed_cids:
                cid_to_inchi[cid] = parts[1]  # 2nd column = InChIKey
                if len(cid_to_inchi) == len(needed_cids):
                    break

    print(f"[OK] InChIKeys found: {len(cid_to_inchi):,}/{len(needed_cids):,}")

    # ---- Step 5: update cache (name -> {CID, SMILES, InChIKey}) ----
    updated = 0
    for norm, original_name in target_set.items():
        cid = found_cids.get(norm)

        if not cid:
            cache[original_name] = {
                "ok": False,
                "cids": [],
                "cid_used": None,
                "smiles": None,
                "inchikey": None,
                "source": "pubchem_offline",
                "note": "no-synonym-match",
            }
            updated += 1
            continue

        sm = cid_to_smiles.get(cid)
        ik = cid_to_inchi.get(cid)

        cache[original_name] = {
            "ok": bool(sm),
            "cids": [cid],
            "cid_used": cid,
            "smiles": sm,
            "inchikey": ik,
            "source": "pubchem_offline",
            "note": "ok" if sm else "no-smiles",
        }
        updated += 1

    save_cache(cache)
    print(f"[DONE] cache saved: {CACHE_JSON}")
    print(f"[INFO] cache entries updated/added this run: {updated:,}")

if __name__ == "__main__":
    main()

[INFO] unique candidate names: 35,220
[INFO] names to resolve (this run): 0
[INFO] nothing to do (all names already cached).


In [3]:
import json
with open("metaspace_images_dump/name_to_pubchem_struct_cache.json", "r") as f:
    data = json.load(f)
data

{'(+)-(S)-Carvone': {'ok': True,
  'cids': [16724],
  'cid_used': 16724,
  'smiles': 'CC1=CC[C@@H](CC1=O)C(=C)C',
  'inchikey': 'InChI=1S/C10H14O/c1-7(2)9-5-4-8(3)10(11)6-9/h4,9H,1,5-6H2,2-3H3/t9-/m0/s1',
  'source': 'pubchem_offline',
  'note': 'ok'},
 '(+)-1(10),4-Cadinadiene': {'ok': True,
  'cids': [441005],
  'cid_used': 441005,
  'smiles': 'CC1=C[C@H]2[C@@H](CCC(=C2CC1)C)C(C)C',
  'inchikey': 'InChI=1S/C15H24/c1-10(2)13-8-6-12(4)14-7-5-11(3)9-15(13)14/h9-10,13,15H,5-8H2,1-4H3/t13-,15-/m0/s1',
  'source': 'pubchem_offline',
  'note': 'ok'},
 '(+)-1(9),10-Pacifigorgiadiene': {'ok': True,
  'cids': [131752220],
  'cid_used': 131752220,
  'smiles': 'CC1CCC2C(CC=C2C1C=C(C)C)C',
  'inchikey': 'InChI=1S/C15H24/c1-10(2)9-15-12(4)5-7-13-11(3)6-8-14(13)15/h8-9,11-13,15H,5-7H2,1-4H3',
  'source': 'pubchem_offline',
  'note': 'ok'},
 '(+)-1,18-Nonacosanediol': {'ok': True,
  'cids': [86172616],
  'cid_used': 86172616,
  'smiles': 'CCCCCCCCCCCC(CCCCCCCCCCCCCCCCCO)O',
  'inchikey': 'InChI=1S/C

In [4]:
# rebuild_parquet_from_cache.py
from pathlib import Path
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

IN_PARQUET  = Path(r"metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates.parquet")
OUT_PARQUET = Path(r"metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates_structures.parquet")
CACHE_JSON  = Path(r"metaspace_images_dump\name_to_pubchem_struct_cache.json")

# ============================================================
# LOAD
# ============================================================
df = pd.read_parquet(IN_PARQUET)
cache = json.loads(CACHE_JSON.read_text(encoding="utf-8"))

# cand_names: "name1; name2; ..." -> ["name1","name2",...]
names_series = (
    df["cand_names"]
    .fillna("")
    .astype(str)
    .map(lambda s: [x.strip() for x in s.split(";") if x.strip()])
)

# ============================================================
# HELPERS
# ============================================================
def _dedup_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

# ============================================================
# ROW BUILDERS (NO NAMES, PURE VALUES)
#   - cand_smiles:      "SMILES ; SMILES ; ..."
#   - cand_inchi:       "InChI=... ; InChI=... ; ..."
#   - cand_pubchem_cids:"CID123 ; CID456 ; ..."
# ============================================================
def row_smiles(names):
    vals = []
    for n in names:
        rec = cache.get(n)
        if not isinstance(rec, dict):
            continue
        cid = rec.get("cid_used")
        smiles = rec.get("smiles")
        if rec.get("ok") and cid and isinstance(smiles, str) and smiles.strip():
            vals.append(smiles.strip())
    return " ; ".join(_dedup_keep_order(vals))

def row_inchi(names):
    vals = []
    for n in names:
        rec = cache.get(n)
        if not isinstance(rec, dict):
            continue
        cid = rec.get("cid_used")
        v = rec.get("inchikey")  # in your cache this is actually an InChI string
        if cid and isinstance(v, str):
            v = v.strip()
            if v.startswith("InChI="):
                vals.append(v)
    return " ; ".join(_dedup_keep_order(vals))

def row_cids(names):
    vals = []
    for n in names:
        rec = cache.get(n)
        if not isinstance(rec, dict):
            continue
        cid = rec.get("cid_used")
        if cid:
            vals.append(f"CID{int(cid)}")
    return " ; ".join(_dedup_keep_order(vals))

# ============================================================
# BUILD COLUMNS WITH tqdm
# ============================================================
tqdm.pandas(desc="Building cand_smiles")
df["cand_smiles"] = names_series.progress_map(row_smiles)

tqdm.pandas(desc="Building cand_inchi")
df["cand_inchi"] = names_series.progress_map(row_inchi)

tqdm.pandas(desc="Building cand_pubchem_cids")
df["cand_pubchem_cids"] = names_series.progress_map(row_cids)

# keep your dtype cleanup
if "save_format" in df.columns:
    df["save_format"] = df["save_format"].astype("string")
if "tiling" in df.columns:
    df["tiling"] = df["tiling"].astype("boolean")

# quick sanity
nonempty_smiles = (df["cand_smiles"].fillna("").astype(str).str.len() > 0).sum()
nonempty_inchi  = (df["cand_inchi"].fillna("").astype(str).str.len() > 0).sum()
nonempty_cids   = (df["cand_pubchem_cids"].fillna("").astype(str).str.len() > 0).sum()
print(f"[INFO] non-empty rows: smiles={nonempty_smiles:,} inchi={nonempty_inchi:,} cids={nonempty_cids:,} / {len(df):,}")

# ============================================================
# STREAMING PARQUET WRITE WITH tqdm
# ============================================================
print("[WRITE] streaming parquet...")

CHUNK = 5000
n_rows = len(df)
n_chunks = (n_rows + CHUNK - 1) // CHUNK

writer = None
for ci in tqdm(range(n_chunks), desc="Writing parquet chunks"):
    start = ci * CHUNK
    end = min(start + CHUNK, n_rows)
    chunk = df.iloc[start:end]

    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT_PARQUET, table.schema)
    writer.write_table(table)

if writer is not None:
    writer.close()

print(f"[DONE] wrote {OUT_PARQUET}")


Building cand_pubchem_cids: 100%|██████████| 695500/695500 [00:06<00:00, 101440.98it/s]


[INFO] non-empty rows: smiles=666,401 inchi=666,401 cids=666,401 / 695,500
[WRITE] streaming parquet...


Writing parquet chunks: 100%|██████████| 140/140 [00:21<00:00,  6.59it/s]

[DONE] wrote metaspace_images_dump\manifest_REBUILT2_expanded_with_candidates_structures.parquet


In [5]:
import pandas as pd
df = pd.read_parquet("metaspace_images_dump/manifest_REBUILT2_expanded_with_candidates_structures.parquet")
df

,dataset_id,name,organism,split,db,fdr,msm,sum_formula,adduct,isotope_index,...,tile_size,tile_stride,Organism_Part,Condition,sum_formula_clean,adduct_clean,cand_names,cand_smiles,cand_inchi,cand_pubchem_cids
0,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,0.0,...,NaN,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)...,CID135398631 ; CID135488904 ; CID135463437
1,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,1.0,...,NaN,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)...,CID135398631 ; CID135488904 ; CID135463437
2,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957401,C10H14N5O8P,-H,2.0,...,NaN,NaN,Colon,"Cancer, xenograft",C10H14N5O8P,-H,Guanosine monophosphate; 8-Oxo-dGMP; Cyclic py...,C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O...,InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)...,CID135398631 ; CID135488904 ; CID135463437
3,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957267,C10H20O2,-H,0.0,...,NaN,NaN,Colon,"Cancer, xenograft",C10H20O2,-H,"(1R,2S,3S,4R)-p-Menthane-2,3-diol; p-Menthane-...",CC1CCC(C(C1O)O)C(C)C ; CC1CCC(C(C1)O)C(C)(C)O ...,InChI=1S/C10H20O2/c1-6(2)8-5-4-7(3)9(11)10(8)1...,CID107175 ; CID556998 ; CID5463962 ; CID12294 ...
4,2016-09-21_16h06m49s,AstraZeneca//CT26_xenograft,Mus musculus (mouse),train,"[HMDB, v4]",0.05,0.957267,C10H20O2,-H,1.0,...,NaN,NaN,Colon,"Cancer, xenograft",C10H20O2,-H,"(1R,2S,3S,4R)-p-Menthane-2,3-diol; p-Menthane-...",CC1CCC(C(C1O)O)C(C)C ; CC1CCC(C(C1)O)C(C)(C)O ...,InChI=1S/C10H20O2/c1-6(2)8-5-4-7(3)9(11)10(8)1...,CID107175 ; CID556998 ; CID5463962 ; CID12294 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695495,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.867454,C45H78O2,+Na,1.0,...,NaN,NaN,Kidney,Control,C45H78O2,+Na,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...,CCCCCCCC/C=C\CCCCCCCC(=O)O[C@H]1CC[C@@]2([C@H]...,InChI=1S/C45H78O2/c1-7-8-9-10-11-12-13-14-15-1...,CID5283632 ; CID53477793
695496,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.867454,C45H78O2,+Na,2.0,...,NaN,NaN,Kidney,Control,C45H78O2,+Na,CE(18:1(9Z)); CE(18:1(11Z)); Palmitoylstigmast...,CCCCCCCC/C=C\CCCCCCCC(=O)O[C@H]1CC[C@@]2([C@H]...,InChI=1S/C45H78O2/c1-7-8-9-10-11-12-13-14-15-1...,CID5283632 ; CID53477793
695497,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.899866,C47H95N2O6P,+Na,0.0,...,NaN,NaN,Kidney,Control,C47H95N2O6P,+Na,SM(d18:0/24:1(15Z)),CCCCCCCCCCCCCCC[C@H]([C@H](COP(=O)([O-])OCC[N+...,InChI=1S/C47H95N2O6P/c1-6-8-10-12-14-16-18-20-...,CID44260133
695498,2026-01-18_22h19m43s,qc_kidney_20260115_bg_entrancelens_40lp_s2,Homo sapiens (human),train,"[HMDB, v4]",0.05,0.899866,C47H95N2O6P,+Na,1.0,...,NaN,NaN,Kidney,Control,C47H95N2O6P,+Na,SM(d18:0/24:1(15Z)),CCCCCCCCCCCCCCC[C@H]([C@H](COP(=O)([O-])OCC[N+...,InChI=1S/C47H95N2O6P/c1-6-8-10-12-14-16-18-20-...,CID44260133


In [6]:
print(df["cand_smiles"][0])
print(df["cand_inchi"][0])
print(df["cand_pubchem_cids"][0])

C1=NC2=C(N1[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)O)O)O)N=C(NC2=O)N ; C1[C@@H]([C@H](O[C@H]1N2C3=C(C(=O)NC(=N3)N)NC2=O)COP(=O)(O)O)O ; C1[C@@H]2[C@@H](C([C@H]3[C@@H](O2)NC4=C(N3)C(=O)NC(=N4)N)(O)O)OP(=O)(O1)O
InChI=1S/C10H14N5O8P/c11-10-13-7-4(8(18)14-10)12-2-15(7)9-6(17)5(16)3(23-9)1-22-24(19,20)21/h2-3,5-6,9,16-17H,1H2,(H2,19,20,21)(H3,11,13,14,18)/t3-,5-,6-,9-/m1/s1 ; InChI=1S/C10H14N5O8P/c11-9-13-7-6(8(17)14-9)12-10(18)15(7)5-1-3(16)4(23-5)2-22-24(19,20)21/h3-5,16H,1-2H2,(H,12,18)(H2,19,20,21)(H3,11,13,14,17)/t3-,4+,5+/m0/s1 ; InChI=1S/C10H14N5O8P/c11-9-14-6-3(7(16)15-9)12-4-8(13-6)22-2-1-21-24(19,20)23-5(2)10(4,17)18/h2,4-5,8,12,17-18H,1H2,(H,19,20)(H4,11,13,14,15,16)/t2-,4-,5+,8-/m1/s1
CID135398631 ; CID135488904 ; CID135463437
